# Phase 2 · Step 3 — Ablation (Sec. 6 item 5) + NF4 memory audit

**Question 1 (Section 4.1 of the report):** why did 4-bit NF4 hurt MMLU (−7.5 pp) much more than GSM8K (−1.5 pp)?
Hypothesis: factual knowledge lives mainly in the feed-forward (MLP) layers, so quantizing *them* is what hurts MMLU.

| Variant | Quantized to NF4 | Kept in FP16 |
|---|---|---|
| `nf4_mlp_only` | feed-forward (MLP) layers | attention layers |
| `nf4_attn_only` | attention layers | feed-forward (MLP) layers |
| `nf4_full` (optional, last) | both — same as Phase 1, but now with per-question logs for paired tests | — |

If the hypothesis is right: `nf4_mlp_only` loses most of the MMLU points, `nf4_attn_only` loses few.

**Question 2 (Section 4.2):** why did the NF4 model hold 4.25 GB right after loading when ~2.7 GB was expected?
Cell 4 audits every tensor of the loaded NF4 model (no benchmark needed, ~3 min).

**How to run:** T4 GPU runtime → **Runtime → Run all**, every session, until cell 2 says everything is done.
Same questions, prompts, scoring and memory measurement as Phase 1. Each variant ≈ 1–1.5 h.

In [ ]:
# ==== 1. Install (the same libraries Phase 1 used) ====
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "bitsandbytes>=0.46.1", "accelerate", "datasets"], check=True)
import torch, transformers, bitsandbytes
print("transformers", transformers.__version__, "| bitsandbytes", bitsandbytes.__version__, "| GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ==== 2. Settings, Drive, progress ====
import os, json, time, re, gc, random, shutil
MODEL_ID = "microsoft/Phi-4-mini-instruct"
ORDER = ["nf4_mlp_only", "nf4_attn_only", "nf4_full"]     # nf4_full is optional (paired-test data)

# Which modules stay in FP16. Patterns work with both transformers 4.x (substring match) and 5.x (regex match).
SKIP = {
    "nf4_mlp_only":  ["lm_head", "self_attn", r".*\.self_attn\..*"],   # quantize MLP only
    "nf4_attn_only": ["lm_head", "mlp", r".*\.mlp\..*"],               # quantize attention only
    "nf4_full":      None,                                              # Phase-1 config
}

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/quant_study_phase2"
except Exception as e:
    print("!! Drive not mounted:", e); DRIVE = "/content/no_drive"
RUN_DIR = f"{DRIVE}/runs"; os.makedirs(RUN_DIR, exist_ok=True)

def n_done(v, t):
    p = f"{RUN_DIR}/{v}__{t}.jsonl"
    return sum(1 for _ in open(p)) if os.path.exists(p) else 0
for v in ORDER:
    print(f"{v:14s} gsm8k {n_done(v,'gsm8k'):3d}/200   mmlu {n_done(v,'mmlu'):3d}/200")
TODO = [v for v in ORDER if min(n_done(v, "gsm8k"), n_done(v, "mmlu")) < 200]
print("\nTo run this session:", TODO if TODO else "NOTHING — ABLATION DONE")

In [ ]:
# ==== 3. Same questions, prompts and scoring as Phase 1 ====
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
random.seed(42)
gsm8k = load_dataset("openai/gsm8k", "main", split="test").shuffle(seed=42).select(range(200))
gsm8k_questions = [{"question": ex["question"], "gold": ex["answer"].split("####")[-1].strip().replace(",", "")} for ex in gsm8k]
mmlu = load_dataset("cais/mmlu", "all", split="test").shuffle(seed=42).select(range(200))
L = ["A", "B", "C", "D"]
mmlu_questions = [{"question": f"{ex['question']}\n" + "\n".join(f"{L[i]}. {c}" for i, c in enumerate(ex["choices"])),
                   "gold": L[ex["answer"]], "subject": ex["subject"]} for ex in mmlu]
assert gsm8k_questions[0]["gold"] == "109" and mmlu_questions[0]["subject"] == "college_physics", "subset differs from Phase 1!"

GSM8K_PROMPT = ("Solve this math problem step by step, then give the final numeric answer "
                "on the last line as 'Answer: <number>'.\n\nProblem: {q}\n")
MMLU_PROMPT  = ("Answer this multiple-choice question. Respond with only the letter "
                "(A, B, C, or D) on the last line as 'Answer: <letter>'.\n\nQuestion: {q}\n")
def extract_number(text):
    text = text.replace(",", ""); m = re.findall(r"Answer:\s*(-?\d+\.?\d*)", text)
    if m: return m[-1]
    nums = re.findall(r"-?\d+\.?\d*", text); return nums[-1] if nums else None
def extract_letter(text):
    m = re.findall(r"Answer:\s*([ABCD])", text.upper())
    if m: return m[-1]
    m = re.findall(r"\b([ABCD])\b", text.upper()); return m[-1] if m else None

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def reset_memory():
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

def load_nf4(skip):
    cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
                             bnb_4bit_use_double_quant=True, llm_int8_skip_modules=skip)
    return AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=cfg, device_map="cuda", dtype=torch.float16)

def count_quantized(model):
    import bitsandbytes as bnb
    c = {"attn_4bit": 0, "mlp_4bit": 0, "attn_fp16": 0, "mlp_fp16": 0}
    for name, m in model.named_modules():
        part = "attn" if ".self_attn." in name + "." else ("mlp" if ".mlp." in name + "." else None)
        if part is None: continue
        if isinstance(m, bnb.nn.Linear4bit): c[f"{part}_4bit"] += 1
        elif isinstance(m, torch.nn.Linear): c[f"{part}_fp16"] += 1
    return c

def make_generate_fn(model, tok):          # identical to Phase 1
    def generate_fn(prompt, max_new_tokens):
        inputs = tok.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True,
                                         return_tensors="pt", return_dict=True).to(model.device)
        in_len = inputs["input_ids"].shape[1]
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
        return tok.decode(out[0][in_len:], skip_special_tokens=True), out.shape[1] - in_len
    return generate_fn
print("ready")

In [ ]:
# ==== 4. NF4 memory audit (answers report Section 4.2; ~3 min, runs once) ====
AUDIT = f"{DRIVE}/report_outputs/nf4_memory_audit.json"
if not os.path.exists(AUDIT):
    reset_memory()
    m = load_nf4(None)
    alloc = torch.cuda.memory_allocated() / 1024**3
    seen, by_kind = set(), {}
    def add(kind, t):
        if t.data_ptr() in seen: return
        seen.add(t.data_ptr()); by_kind[kind] = by_kind.get(kind, 0) + t.numel() * t.element_size()
    for name, p in m.named_parameters():
        kind = ("embedding" if "embed_tokens" in name else "lm_head" if "lm_head" in name
                else "linear_4bit_packed" if p.dtype == torch.uint8 else f"other_{str(p.dtype).replace('torch.', '')}")
        add(kind, p)
    for name, b in m.named_buffers():
        add("buffers", b)
    for name, mod in m.named_modules():           # NF4 scale tensors live in quant_state, not in parameters
        qs = getattr(getattr(mod, "weight", None), "quant_state", None)
        if qs is not None:
            for t in [qs.absmax, getattr(qs, "code", None)] + ([qs.state2.absmax, qs.state2.code] if getattr(qs, "nested", False) else []):
                if isinstance(t, torch.Tensor): add("nf4_scales", t)
    tied = m.lm_head.weight.data_ptr() == m.model.embed_tokens.weight.data_ptr()
    audit = {"allocated_after_load_gb": round(alloc, 3), "lm_head_tied_to_embedding": tied,
             "bytes_by_kind_gb": {k: round(v / 1024**3, 3) for k, v in sorted(by_kind.items(), key=lambda x: -x[1])},
             "sum_of_tensors_gb": round(sum(by_kind.values()) / 1024**3, 3), "transformers": transformers.__version__}
    os.makedirs(os.path.dirname(AUDIT), exist_ok=True); json.dump(audit, open(AUDIT, "w"), indent=1)
    del m; reset_memory()
print(json.dumps(json.load(open(AUDIT)), indent=1))

In [ ]:
# ==== 5. Run the ablation variants (resumable; one JSON line per question on Drive) ====
@torch.no_grad()
def run_resumable(variant, gen, task, questions):
    path = f"{RUN_DIR}/{variant}__{task}.jsonl"
    done = {json.loads(l)["idx"]: json.loads(l) for l in open(path)} if os.path.exists(path) else {}
    todo = [i for i in range(len(questions)) if i not in done]
    print(f"[{variant}|{task}] {len(done)} done, {len(todo)} to go")
    with open(path, "a") as f:
        for n, i in enumerate(todo):
            ex = questions[i]
            prompt = (GSM8K_PROMPT if task == "gsm8k" else MMLU_PROMPT).format(q=ex["question"])
            t0 = time.time(); text, n_tok = gen(prompt, 512); dt = time.time() - t0
            pred = extract_number(text) if task == "gsm8k" else extract_letter(text)
            rec = {"idx": i, "gold": ex["gold"], "pred": pred,
                   "correct": (pred is not None) and str(pred).strip() == str(ex["gold"]).strip(),
                   "n_tokens": int(n_tok), "wall_s": dt, "subject": ex.get("subject"), "output": text}
            f.write(json.dumps(rec) + "\n"); f.flush(); done[i] = rec
            if (n + 1) % 20 == 0:
                print(f"  ...{len(done)}/200  running acc {100*sum(r['correct'] for r in done.values())/len(done):.1f}%")
    recs = list(done.values())
    print(f"[{variant}|{task}] acc={100*sum(r['correct'] for r in recs)/len(recs):.1f}%  "
          f"tok/s={sum(r['n_tokens'] for r in recs)/sum(r['wall_s'] for r in recs):.1f}")

for v in TODO:
    print(f"\n==================== {v} ====================")
    reset_memory()
    model = load_nf4(SKIP[v])
    counts = count_quantized(model)
    print("quantized layers:", counts)        # proves which half is 4-bit
    if v == "nf4_mlp_only":  assert counts["attn_4bit"] == 0 and counts["mlp_4bit"] > 0, "wrong layers quantized — send this to Claude"
    if v == "nf4_attn_only": assert counts["mlp_4bit"] == 0 and counts["attn_4bit"] > 0, "wrong layers quantized — send this to Claude"
    load_gb = torch.cuda.memory_allocated() / 1024**3
    gen = make_generate_fn(model, tokenizer)
    for task, qs in [("gsm8k", gsm8k_questions), ("mmlu", mmlu_questions)]:
        run_resumable(v, gen, task, qs)
    meta_p = f"{RUN_DIR}/{v}__meta.json"
    meta = json.load(open(meta_p)) if os.path.exists(meta_p) else {}
    meta.update({"runtime": "transformers+bnb", "weights": v, "kv": "f16", "quantized_layers": counts,
                 "allocated_after_load_gb": round(load_gb, 3),
                 "peak_mem_gb": round(max(torch.cuda.max_memory_allocated() / 1024**3, meta.get("peak_mem_gb", 0)), 3)})
    json.dump(meta, open(meta_p, "w"), indent=1); print(meta)
    del model, gen; reset_memory()
print("\nAll selected ablation variants finished." if TODO else "Nothing to do.")